# MapBiomas Chile Collection 2.0 — GEE exploration (test cities)

Pull LULC class histograms from Google Earth Engine, then join with OCHA comuna polygons to get land-use shares per city.

**Scope:** 12 HIAP-MEED Los Ríos test cities only (locodes from IPCC feasibility `socioeconomic_indicators.csv`). Every indicator in `indicator_thresholds.csv` is emitted per comuna; **no coverage → `area_pct = 0`, `attribute_category = very low`** (never dropped).

Prerequisites: [Earth Engine signup](https://earthengine.google.com/), `earthengine authenticate` once, `pip install earthengine-api geopandas pandas`.

In [61]:
from pathlib import Path

ASSET_ID = (
    "projects/mapbiomas-chile/assets/LULC/COLLECTION-02/CLASSIFICATIONS/"
    "classification-final/clasificacion-final-2"
)
START_YEAR = 1999
YEAR = 2023
SCALE_M = 30
OCHA_LAYER = "cl_admin_locode"


def _find_release_dir() -> Path:
    """Locate collection-02 regardless of notebook kernel cwd."""
    rel_paths = (
        Path("reviews/gee/cl-mapbiomas/releases/collection-02"),
        Path("gee/cl-mapbiomas/releases/collection-02"),
        Path("dataset-review/reviews/gee/cl-mapbiomas/releases/collection-02"),
    )
    for base in (Path.cwd(), *Path.cwd().parents):
        if base.name == "collection-02" and (base / "review.yaml").is_file():
            return base
        for rel in rel_paths:
            candidate = (base / rel).resolve()
            if (candidate / "review.yaml").is_file():
                return candidate
    raise FileNotFoundError("collection-02 release folder not found (review.yaml)")


def _find_ocha_gpkg() -> Path:
    rel = Path(
        "ocha-rolac/cl-ocha-ab/releases/2021/sample/raw_data_cl_ocha_ab.gpkg"
    )
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            base / rel,
            base / "reviews" / rel,
            base / "dataset-review" / "reviews" / rel,
        ):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(f"OCHA GPKG not found: {rel}")


def _find_test_locodes_csv() -> Path:
    rel = Path(
        "ipcc/ipcc-sr15-mitigation-feasibility/releases/2018/sample/"
        "socioeconomic_indicators.csv"
    )
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            base / rel,
            base / "reviews" / rel,
            base / "dataset-review" / "reviews" / rel,
        ):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(f"Test-city locodes CSV not found: {rel}")


RELEASE_DIR = _find_release_dir()
DATA_DIR = RELEASE_DIR / "data"
SAMPLE_DIR = RELEASE_DIR / "sample"
OCHA_GPKG = _find_ocha_gpkg()
TEST_LOCODES_CSV = _find_test_locodes_csv()
OUTPUT_SUFFIX = "_hiap_meed"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

In [62]:
import json
import ee
import geopandas as gpd
import pandas as pd

In [63]:
ee.Initialize(project="citycatalyst")

In [64]:
lulc_stack = ee.Image(ASSET_ID)

In [65]:
band_index = YEAR - START_YEAR
lulc_year = lulc_stack.select([band_index]).rename("lulc")

In [66]:
from shapely import make_valid

cities = gpd.read_file(OCHA_GPKG, layer=OCHA_LAYER)
cities = cities.to_crs(4326)

invalid = ~cities.geometry.is_valid
if invalid.any():
    cities.loc[invalid, "geometry"] = cities.loc[invalid, "geometry"].apply(make_valid)

test_cities = (
    pd.read_csv(TEST_LOCODES_CSV)[
        ["locode", "comuna", "region", "region_nombre", "comuna_nombre"]
    ]
    .drop_duplicates("locode")
    .assign(
        comuna_code=lambda d: "CL" + d["comuna"].astype(str),
        region_code=lambda d: "CL" + d["region"].astype(str),
    )
)
TEST_LOCODES = test_cities["locode"].tolist()
N_TEST_CITIES = len(test_cities)

cities_gee = cities[cities["locode"].isin(TEST_LOCODES)].copy()
missing_gee = sorted(set(TEST_LOCODES) - set(cities_gee["locode"]))
if missing_gee:
    print(
        f"GEE zonal: {len(cities_gee)}/{N_TEST_CITIES} comunas with OCHA polygons; "
        f"missing (all indicators → 0%): {', '.join(missing_gee)}"
    )
else:
    print(f"GEE zonal: {N_TEST_CITIES} comunas")
cities = cities_gee

GEE zonal: 10/12 comunas with OCHA polygons; missing (all indicators → 0%): CL MRQ, CL RBU


In [67]:
prop_cols = ["comuna_code", "region_code", "comuna_name", "locode", "locode_name"]
cities_geojson = json.loads(cities[prop_cols + ["geometry"]].to_json())
cities_fc = ee.FeatureCollection(cities_geojson)

In [68]:
zonal = lulc_year.reduceRegions(
    collection=cities_fc,
    reducer=ee.Reducer.frequencyHistogram(),
    scale=SCALE_M,
    tileScale=4,
)

In [69]:
raw_path = SAMPLE_DIR / f"lulc_histogram_{YEAR}{OUTPUT_SUFFIX}.geojson"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(zonal.getInfo(), f)

In [70]:
with open(raw_path, encoding="utf-8") as f:
    zonal_geojson = json.load(f)

rows = []
for feat in zonal_geojson["features"]:
    props = feat["properties"]
    hist = props.get("histogram") or {}
    for class_id, pixel_count in hist.items():
        rows.append(
            {
                "year": YEAR,
                "comuna_code": props.get("comuna_code"),
                "locode": props.get("locode"),
                "lulc_class": int(class_id),
                "pixel_count": int(pixel_count),
            }
        )

lulc_long = pd.DataFrame(rows)

In [71]:
totals = lulc_long.groupby("comuna_code", as_index=False)["pixel_count"].sum()
totals = totals.rename(columns={"pixel_count": "pixel_total"})

lulc_pct = lulc_long.merge(totals, on="comuna_code", how="left")
lulc_pct["share"] = lulc_pct["pixel_count"] / lulc_pct["pixel_total"]

In [72]:
crosswalk = pd.read_csv(DATA_DIR / "lulc_to_indicator_crosswalk.csv")
threshold_df = pd.read_csv(DATA_DIR / "indicator_thresholds.csv")
INDICATOR_GROUPS = sorted(
    {ind.removesuffix("_share") for ind in threshold_df["indicator"].unique()}
)
comuna_names = test_cities[["comuna_code", "comuna_nombre"]].rename(
    columns={"comuna_nombre": "comuna_name"}
)

observed = (
    lulc_pct.merge(crosswalk[["lulc_class", "indicator_group"]], on="lulc_class")
    .groupby(["year", "comuna_code", "indicator_group"], as_index=False, dropna=False)
    .agg(pixel_count=("pixel_count", "sum"), pixel_total=("pixel_total", "first"))
)
observed["area_pct"] = (
    100 * observed["pixel_count"] / observed["pixel_total"]
).round(1)

# Full grid: every test comuna × every thresholded indicator (missing → 0%)
full_grid = comuna_names.assign(_k=1).merge(
    pd.DataFrame({"indicator_group": INDICATOR_GROUPS, "_k": 1}),
    on="_k",
).drop(columns="_k")

lulc_indicators = (
    full_grid.merge(
        observed[["comuna_code", "indicator_group", "area_pct"]],
        on=["comuna_code", "indicator_group"],
        how="left",
    )
    .assign(year=YEAR)
)
lulc_indicators["area_pct"] = lulc_indicators["area_pct"].fillna(0).round(1)

out_cols = ["comuna_code", "comuna_name", "year", "indicator_group", "area_pct"]
lulc_indicators[out_cols].to_csv(
    DATA_DIR / f"city_landuse_indicators{OUTPUT_SUFFIX}.csv",
    index=False,
)
lulc_indicators[out_cols].to_csv(
    SAMPLE_DIR / f"lulc_indicators_by_city_{YEAR}{OUTPUT_SUFFIX}.csv",
    index=False,
)

In [73]:
# Optional: MapBiomas class-level detail (with labels)
legend = pd.read_csv(DATA_DIR / "legend_collection_02.csv")
legend["lulc_label"] = (
    legend["class_level_3_en"].replace("", pd.NA)
    .fillna(legend["class_level_2_en"].replace("", pd.NA))
    .fillna(legend["class_level_1_en"])
)
lulc_pct.merge(legend[["lulc_class", "lulc_label"]], on="lulc_class").merge(
    comuna_names, on="comuna_code"
).to_csv(SAMPLE_DIR / f"lulc_share_by_city_{YEAR}{OUTPUT_SUFFIX}.csv", index=False)

In [74]:
def bucket_for(threshold_df, indicator, value):
    value = float(value)
    if value == 0.0:
        return "very_low"
    sub = threshold_df[threshold_df.indicator == indicator].sort_values("lower_pct")
    for _, row in sub.iterrows():
        if row.bucket == "very_high" and value >= row.lower_pct:
            return row.bucket
        if row.lower_pct <= value < row.upper_pct:
            return row.bucket
    return "very_low"


def bucket_label(bucket):
    return bucket.replace("_", " ")


city_geo = test_cities.assign(
    region=lambda d: d["region"].astype(int),
    comuna=lambda d: d["comuna"].astype(int),
)

raw_rows = []
for _, row in lulc_indicators.iterrows():
    attribute_type = f"{row['indicator_group']}_share"
    bucket = bucket_for(threshold_df, attribute_type, row["area_pct"])
    geo = city_geo.loc[city_geo["comuna_code"] == row["comuna_code"]].iloc[0]
    raw_rows.append(
        {
            "region": geo["region"],
            "comuna": geo["comuna"],
            "region_nombre": geo["region_nombre"],
            "comuna_nombre": geo["comuna_nombre"],
            "attribute_type": attribute_type,
            "attribute_value": row["area_pct"],
            "attribute_units": "percent",
            "attribute_category": bucket_label(bucket),
        }
    )

raw_data_cl_mapbiomas_lulc = pd.DataFrame(raw_rows)
assert len(raw_data_cl_mapbiomas_lulc) == N_TEST_CITIES * len(INDICATOR_GROUPS), (
    f"expected {N_TEST_CITIES}×{len(INDICATOR_GROUPS)} rows, "
    f"got {len(raw_data_cl_mapbiomas_lulc)}"
)
raw_data_cl_mapbiomas_lulc.to_csv(
    DATA_DIR / f"raw_data_cl_mapbiomas_lulc{OUTPUT_SUFFIX}.csv",
    index=False,
)
print(
    f"Wrote {len(raw_data_cl_mapbiomas_lulc)} rows "
    f"({N_TEST_CITIES} cities × {len(INDICATOR_GROUPS)} indicators)"
)

Wrote 156 rows (12 cities × 13 indicators)
